# Public Interest Data Collection — NASCAR Sponsor Mentions

**Data source:** Google Trends (pytrends)

Reddit's public JSON API returned 403 (blocked at IP level) despite correct User-Agent headers.
Per mentor guidance, we use **Google Trends** as the public interest proxy — it measures relative
weekly search interest (0–100) for each sponsor in the context of NASCAR, which serves the same
role in the visibility model as Reddit mention counts.

Output is saved as `reddit_mentions.csv` for compatibility with downstream modules.

## Step 1: Set Up Google Trends (pytrends)

In [35]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytrends', '-q'])

from pytrends.request import TrendReq
import time

pytrends = TrendReq(hl='en-US', tz=360)

print("Testing Google Trends connection...")
try:
    pytrends.build_payload(['NASCAR'], timeframe='2024-02-18 2024-11-10', geo='US')
    data = pytrends.interest_over_time()
    if not data.empty:
        print(f"Connected! Got {len(data)} weeks of data")
        print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")
    else:
        print("Connected but no data returned — try again in a moment.")
except Exception as e:
    print(f"Error: {e}")

Testing Google Trends connection...
Connected! Got 267 weeks of data
Date range: 2024-02-18 to 2024-11-10


## Step 2: Define Sponsors and Search Terms

In [36]:
import pandas as pd
from datetime import datetime
import time

# Sponsor search terms — "[Sponsor] NASCAR" gives sport-specific interest
target_sponsors = {
    'FedEx': 'FedEx NASCAR',
    'NAPA Auto Parts': 'NAPA NASCAR',
    "McDonald's": "McDonald's NASCAR",
    "Love's Travel Stops": "Loves Travel Stops NASCAR",
}

START_DATE = '2024-02-18'
END_DATE = '2024-11-10'
START_DT = datetime(2024, 2, 18)
END_DT = datetime(2024, 11, 10)

print(f"Search period: {START_DATE} to {END_DATE}")
print(f"Target sponsors: {list(target_sponsors.keys())}")

Search period: 2024-02-18 to 2024-11-10
Target sponsors: ['FedEx', 'NAPA Auto Parts', "McDonald's", "Love's Travel Stops"]


## Step 3: Build the Interest Collection Function

In [37]:
def collect_trends_for_sponsor(sponsor_name, search_term, timeframe):
    """
    Collect Google Trends weekly interest for a sponsor search term.
    Returns a list of dicts with week_date and interest_score (0-100).
    """
    results = []
    try:
        pytrends.build_payload([search_term], timeframe=timeframe, geo='US')
        data = pytrends.interest_over_time()

        if data.empty:
            print(f"  No data returned for '{search_term}'")
            return results

        for date, row in data.iterrows():
            results.append({
                'sponsor': sponsor_name,
                'search_term': search_term,
                'week_date': date,
                'interest_score': int(row[search_term]),
            })

        print(f"  {len(results)} weeks · peak interest: {data[search_term].max()} · avg: {data[search_term].mean():.1f}")

    except Exception as e:
        print(f"  Error: {e}")

    return results

### Test on One Sponsor

In [38]:
print("Testing Google Trends collection for FedEx...")
timeframe = f"{START_DATE} {END_DATE}"
test_results = collect_trends_for_sponsor('FedEx', target_sponsors['FedEx'], timeframe)

if test_results:
    test_df = pd.DataFrame(test_results)
    print(f"\nSample data:")
    print(test_df.head(10).to_string(index=False))
else:
    print("No data returned — check your internet connection.")

Testing Google Trends collection for FedEx...
  267 weeks · peak interest: 100 · avg: 0.9

Sample data:
sponsor  search_term  week_date  interest_score
  FedEx FedEx NASCAR 2024-02-18               0
  FedEx FedEx NASCAR 2024-02-19             100
  FedEx FedEx NASCAR 2024-02-20               0
  FedEx FedEx NASCAR 2024-02-21               0
  FedEx FedEx NASCAR 2024-02-22               0
  FedEx FedEx NASCAR 2024-02-23               0
  FedEx FedEx NASCAR 2024-02-24               0
  FedEx FedEx NASCAR 2024-02-25               0
  FedEx FedEx NASCAR 2024-02-26               0
  FedEx FedEx NASCAR 2024-02-27               0


## Step 4: Collect Data for All Sponsors

In [39]:
print("Collecting Google Trends data for all sponsors...")
print("=" * 50)

timeframe = f"{START_DATE} {END_DATE}"
all_trends = []

for i, (sponsor, search_term) in enumerate(target_sponsors.items(), 1):
    print(f"\n[{i}/{len(target_sponsors)}] {sponsor} → '{search_term}'")
    results = collect_trends_for_sponsor(sponsor, search_term, timeframe)
    all_trends.extend(results)

    if i < len(target_sponsors):
        time.sleep(5)  # Rate limit: Google Trends allows ~1 request/5s

trends_df = pd.DataFrame(all_trends)
trends_df['week_date'] = pd.to_datetime(trends_df['week_date'])

print("\n" + "=" * 50)
print(f"Collection complete! {len(trends_df)} weekly data points")
print("\nAverage interest score by sponsor:")
print(trends_df.groupby('sponsor')['interest_score'].agg(['mean', 'max']).round(1))


[1/4] FedEx → 'FedEx NASCAR'
  267 weeks · peak interest: 100 · avg: 0.9

[2/4] NAPA Auto Parts → 'NAPA NASCAR'
  267 weeks · peak interest: 100 · avg: 1.0

[3/4] McDonald's → 'McDonald's NASCAR'
  267 weeks · peak interest: 100 · avg: 0.4

[4/4] Love's Travel Stops → 'Loves Travel Stops NASCAR'
  267 weeks · peak interest: 100 · avg: 0.6

Collection complete! 1068 weekly data points

Average interest score by sponsor:
                     mean  max
sponsor                       
FedEx                 0.9  100
Love's Travel Stops   0.6  100
McDonald's            0.4  100
NAPA Auto Parts       1.0  100


## Step 5: Assign to Race Periods & Aggregate

In [40]:
print(f"Data covers: {trends_df['week_date'].min().date()} to {trends_df['week_date'].max().date()}")
print(f"Total weekly data points: {len(trends_df)}")
print("\nInterest score by sponsor:")
print(trends_df.groupby('sponsor')['interest_score'].describe().round(1))

Data covers: 2024-02-18 to 2024-11-10
Total weekly data points: 1068

Interest score by sponsor:
                     count  mean  std  min  25%  50%  75%    max
sponsor                                                         
FedEx                267.0   0.9  8.8  0.0  0.0  0.0  0.0  100.0
Love's Travel Stops  267.0   0.6  7.5  0.0  0.0  0.0  0.0  100.0
McDonald's           267.0   0.4  6.1  0.0  0.0  0.0  0.0  100.0
NAPA Auto Parts      267.0   1.0  8.7  0.0  0.0  0.0  0.0  100.0


In [41]:
# Load race schedule and build approximate race dates
from datetime import timedelta

race_schedule_path = '../data/processed/race_results_2024_clean.csv'
race_df = pd.read_csv(race_schedule_path)

season_start = datetime(2024, 2, 18)
season_end = datetime(2024, 11, 10)
total_days = (season_end - season_start).days
max_race = race_df['Race_Number'].max()

race_dates_2024 = {
    r: (season_start + timedelta(days=total_days * (r - 1) / (max_race - 1))).strftime('%Y-%m-%d')
    for r in range(1, max_race + 1)
}

race_df['Race_Date'] = pd.to_datetime(race_df['Race_Number'].map(race_dates_2024))
race_dates = race_df.groupby('Race_Name')['Race_Date'].first().sort_values()
print(f"Race schedule loaded: {len(race_dates)} races")


def assign_race_period(week_date, race_dates):
    """Assign a week to the most recent race that preceded it."""
    prior_races = race_dates[race_dates <= week_date]
    if len(prior_races) == 0:
        return "Pre-Season"
    return prior_races.index[-1]


trends_df['race_period'] = trends_df['week_date'].apply(
    lambda x: assign_race_period(x, race_dates)
)

print("\nWeeks by race period (top 10):")
print(trends_df['race_period'].value_counts().head(10))

Race schedule loaded: 26 races

Weeks by race period (top 10):
race_period
Michigan           124
Watkins Glen       120
Homestead           68
Indianapolis        60
Charlotte Roval     60
Atlanta             32
Phoenix             32
Bristol             32
Richmond            32
Texas               32
Name: count, dtype: int64


### Create Aggregated Summary by Sponsor and Race

In [42]:
# Aggregate weekly interest by sponsor and race period
reddit_summary = trends_df.groupby(['sponsor', 'race_period']).agg(
    mention_count=('interest_score', 'count'),    # weeks in this period
    total_score=('interest_score', 'sum'),        # cumulative interest
    avg_score=('interest_score', 'mean'),         # average weekly interest
    avg_comments=('interest_score', 'min'),       # placeholder (not applicable)
).round(2).reset_index()

reddit_summary = reddit_summary.rename(columns={'race_period': 'race_name'})
reddit_summary['total_comments'] = 0
reddit_summary['avg_comments'] = 0
reddit_summary['engagement_per_mention'] = reddit_summary['avg_score']
reddit_summary['data_source'] = 'google_trends'

# Add race number
race_number_map = race_df.groupby('Race_Name')['Race_Number'].first()
reddit_summary['race_number'] = reddit_summary['race_name'].map(race_number_map)

print(f"Summary: {len(reddit_summary)} sponsor-race rows")
print("\nSponsor-Race Interest Summary (sample):")
print(reddit_summary[['sponsor', 'race_name', 'mention_count', 'avg_score', 'total_score']].head(15))

Summary: 104 sponsor-race rows

Sponsor-Race Interest Summary (sample):
   sponsor              race_name  mention_count  avg_score  total_score
0    FedEx                Atlanta              8       0.00            0
1    FedEx                Bristol              8       0.00            0
2    FedEx                   COTA              7       0.00            0
3    FedEx              Charlotte              8       0.00            0
4    FedEx        Charlotte Roval             15       0.00            0
5    FedEx  Chicago Street Course              8       0.00            0
6    FedEx             Darlington              7       0.00            0
7    FedEx                Daytona              7      14.29          100
8    FedEx                  Dover              7       0.00            0
9    FedEx                Gateway              8       0.00            0
10   FedEx              Homestead             17       0.00            0
11   FedEx           Indianapolis             15    

## Step 6: Save Data

In [43]:
import json
from datetime import datetime as dt

# Save raw weekly trends
trends_df.to_csv('../data/raw/reddit_mentions_2024_raw.csv', index=False)
print("Saved raw weekly trends: reddit_mentions_2024_raw.csv")

# Save aggregated summary
reddit_summary.to_csv('../data/processed/reddit_mentions_2024.csv', index=False)
print("Saved aggregated summary: reddit_mentions_2024.csv")

# Save final deliverable
reddit_summary.to_csv('../data/processed/reddit_mentions.csv', index=False)
print("Saved final deliverable: reddit_mentions.csv")

# Save metadata
metadata = {
    'collection_date': dt.now().isoformat(),
    'data_source': 'Google Trends (pytrends) — Reddit API returned 403',
    'sponsors': list(target_sponsors.keys()),
    'search_terms': list(target_sponsors.values()),
    'date_range': f"{START_DATE} to {END_DATE}",
    'total_weekly_points': len(trends_df),
    'note': 'Per mentor guidance: Google Trends used as public interest proxy'
}
with open('../data/processed/reddit_collection_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata: reddit_collection_metadata.json")

print(f"\n=== Public Interest Data Collection Complete ===")
print(f"  Source: Google Trends")
print(f"  Output: reddit_mentions.csv ({len(reddit_summary)} sponsor-race rows)")

Saved raw weekly trends: reddit_mentions_2024_raw.csv
Saved aggregated summary: reddit_mentions_2024.csv
Saved final deliverable: reddit_mentions.csv
Saved metadata: reddit_collection_metadata.json

=== Public Interest Data Collection Complete ===
  Source: Google Trends
  Output: reddit_mentions.csv (104 sponsor-race rows)
